In [3]:
# Importing Libraries
import pandas as pd
import pyodbc
import numpy as np
from src.utils.db_connector import get_sqlalchemy_engine
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report
import joblib

In [4]:
# Data Importing
ML_TABLE = "gold.T_DIM_CUSTOMER_HABIT"
try:
    Engine=get_sqlalchemy_engine()
    sql_query=f"SELECT * FROM {ML_TABLE}"
    df_ml = pd.read_sql(sql_query, Engine)
except Exception as e:
    print("Error Raised : ", e)

In [11]:
# Data Spliting
conditions = [
    (df_ml['avg_customer_review'] >= 4.5),
    (df_ml['avg_customer_review'] >= 3.5)
]
choices = [2, 1]
df_ml['Y_satisfaction_flag'] = np.select(
    conditions,
    choices,
    default=0
)
Y = df_ml['Y_satisfaction_flag']
X = df_ml.drop(columns=['customer_unique_id', 'avg_customer_review', 'Y_satisfaction_flag'])
print(f"Target Y successfully defined as multi-class (0, 1, 2). Target size: {Y.shape}")
print(f"Target X successfully defined as multi-class (0, 1, 2). Target size: {X.shape}")

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42,
    stratify=Y
)
print(f"Training set size: {X_train.shape[0]:,} samples")
print(f"Testing set size: {X_test.shape[0]:,} samples")


Target Y successfully defined as multi-class (0, 1, 2). Target size: (94029,)
Target X successfully defined as multi-class (0, 1, 2). Target size: (94029, 11)
Training set size: 75,223 samples
Testing set size: 18,806 samples


In [12]:
X.count()

customer_city               94029
customer_state              94029
customer_lat                94029
customer_lng                94029
recency_days                94029
total_orders                94029
total_items_purchased       94029
total_revenue_spent         94029
max_payment_installments    94029
avg_payment_types_used      94029
Y_satisfaction_class        94029
dtype: int64